<a href="https://colab.research.google.com/github/farezae/mscproj/blob/main/AMT_logits_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt install fluidsynth

!git clone https://github.com/jthickstun/anticipation.git
!pip install ./anticipation
!pip install -r anticipation/requirements.txt
!pip install matplotlib
!pip install torch tqdm

In [25]:
import math
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import sys,time

import torch
import torch.nn.functional as F
import midi2audio
import transformers

from tqdm import tqdm

from anticipation import ops
from anticipation.config import *
from anticipation.vocab import *
from transformers import AutoModelForCausalLM
from pathlib import Path
from IPython.display import Audio


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SMALL_MODEL = 'stanford-crfm/music-small-800k'     # faster inference, worse sample quality
MEDIUM_MODEL = 'stanford-crfm/music-medium-800k'   # slower inference, better sample quality
LARGE_MODEL = 'stanford-crfm/music-large-800k'     # slowest inference, best sample quality

# load an anticipatory music transformer
model = AutoModelForCausalLM.from_pretrained(SMALL_MODEL).cuda()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/512M [00:00<?, ?B/s]

Some weights of the model checkpoint at stanford-crfm/music-small-800k were not used when initializing GPT2LMHeadModel: ['token_out_embeddings']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [41]:
def safe_logits(logits, idx):
    logits[CONTROL_OFFSET:SPECIAL_OFFSET] = -float('inf') # don't generate controls
    logits[SPECIAL_OFFSET:] = -float('inf')               # don't generate special tokens

    # don't generate stuff in the wrong time slot
    if idx % 3 == 0:
        logits[DUR_OFFSET:DUR_OFFSET+MAX_DUR] = -float('inf')
        logits[NOTE_OFFSET:NOTE_OFFSET+MAX_NOTE] = -float('inf')
    elif idx % 3 == 1:
        logits[TIME_OFFSET:TIME_OFFSET+MAX_TIME] = -float('inf')
        logits[NOTE_OFFSET:NOTE_OFFSET+MAX_NOTE] = -float('inf')
    elif idx % 3 == 2:
        logits[TIME_OFFSET:TIME_OFFSET+MAX_TIME] = -float('inf')
        logits[DUR_OFFSET:DUR_OFFSET+MAX_DUR] = -float('inf')

    return logits

def future_logits(logits, curtime):
    """ don't sample events in the past """
    if curtime > 0:
        logits[TIME_OFFSET:TIME_OFFSET+curtime] = -float('inf')
    return logits

def instr_logits(logits, full_history):
    """ don't sample more than 16 instruments """
    instrs = ops.get_instruments(full_history)
    if len(instrs) < 15: # 16 - 1 to account for the reserved drum track
        return logits

    for instr in range(MAX_INSTR):
        if instr not in instrs:
            logits[NOTE_OFFSET+instr*MAX_PITCH:NOTE_OFFSET+(instr+1)*MAX_PITCH] = -float('inf')

    return logits

In [42]:
z = [AUTOREGRESS]
def get_logits(model, z, tokens,position=None, current_time=None):
    device = model.device
    model.eval()
    logits_list=[]
    new_token=[]


    assert len(tokens) % 3 == 0

    history = tokens.copy()
    lookback = max(len(tokens) - 1017, 0)
    history = history[lookback:] # Markov window

    offset = ops.min_time(history, seconds=False)
    history[::3] = [tok - offset for tok in history[::3]] # relativize time in the history buffer

    with torch.no_grad():
        for i in range(3):
            input_tokens = torch.tensor(z + history + new_token).unsqueeze(0).to(model.device)
            logits = model(input_tokens).logits[0,-1]

            idx = input_tokens.shape[1]-1
            logits = safe_logits(logits, idx)

            if i == 0 and current_time is not None:
                logits = future_logits(logits, current_time - offset)
            elif i == 2:
                logits = instr_logits(logits, tokens)

            logits_list.append(logits.cpu())
    return logits_list

In [8]:
# get saved tokenisations
folder_path = '/content/drive/MyDrive/MusicData/lakh_generated_events'
file_list = os.listdir(folder_path)


In [53]:
segment_length = 10
all_segment_embeddings = []
all_final_logits = []

# break each tokenisation into 'segments'
for filename in file_list[:1]:
    file_path = os.path.join(folder_path, filename)
    try:
      with open(file_path, 'r') as f:
          print (" filename: ", filename)

          # extract the token sequence
          tokens = [int(line.strip()) for line in f if line.strip()]
          print (" tokens: ", tokens)

          # segment the token sequence
          total_duration = ops.max_time(tokens)
          start_time = 0
          while start_time < total_duration:
            end_time = start_time + segment_length
            segment = ops.clip(tokens, start_time, end_time)

            if not segment:
              start_time = end_time
              continue  # skip any empty segments
            print (" segment: ", segment)

            segment_tensor = torch.tensor(segment).unsqueeze(0)   # works
            long_segment_tensor = segment_tensor.long()
            print(f"Type of segment_tensor: {segment_tensor.dtype}")
            print(f"Size (shape) of segment_tensor: {segment_tensor.shape}")

            final_segment_tensor = segment_tensor.to(model.device) # does not work

            with torch.no_grad():
              outputs = model(segment_tensor, output_hidden_states=True)
              all_hidden_states = outputs.hidden_states

              last_layer_output = all_hidden_states[-1]
              segment_embedding= torch.mean(last_layer_output, dim=1)
              #final_logits = get_logits(model, z, segment)

              print (" segment embedding (based on last layer): ", segment_embedding)
              #print (" final logits: ", final_logits)

              all_segment_embeddings.append(segment_embedding)
              #all_final_logits.append(final_logits)

            start_time = end_time

    except Exception as e:
      print(f"Error processing file {filename}: {e}")
      continue



 filename:  Vive_chi_vive_events.txt
 tokens:  [319, 10008, 27422, 345, 10006, 27420, 358, 10006, 27420, 372, 10008, 27422, 425, 10054, 11079, 425, 10062, 11067, 425, 10011, 15263, 425, 10164, 14519, 425, 10169, 14514, 425, 10054, 22471, 425, 10062, 22459, 425, 10012, 27430, 425, 10009, 27420, 425, 10169, 14507, 427, 10165, 14523, 451, 10010, 15263, 451, 10006, 21431, 451, 10006, 21435, 451, 10006, 21426, 451, 10008, 27430, 465, 10007, 21435, 465, 10007, 21426, 465, 10005, 21431, 478, 10010, 15263, 478, 10010, 27430, 478, 10009, 27422, 504, 10022, 11080, 504, 10033, 11068, 504, 10009, 15263, 504, 10022, 22472, 504, 10033, 22460, 504, 10007, 21431, 504, 10008, 21435, 504, 10006, 21426, 504, 10010, 27430, 518, 10006, 21431, 518, 10006, 21426, 518, 10005, 21435, 531, 10026, 11067, 531, 10026, 11079, 531, 10010, 15263, 531, 10026, 22459, 531, 10026, 22471, 531, 10006, 27430, 531, 10009, 27420, 558, 10014, 11063, 558, 10013, 11075, 558, 10009, 15263, 558, 10014, 22455, 558, 10013, 22467, 55

In [50]:
# Check the model's vocabulary size
print(f"Model vocabulary size: {model.config.vocab_size}")

# Check the maximum token ID in your segment
max_token_id = max(segment)
print(f"Maximum token ID in segment: {max_token_id}")

Model vocabulary size: 55028
Maximum token ID in segment: 27430
